# 03 · Under the hood: snapshots, manifests and files

**Time:** ~25 min · **You will learn:** how Iceberg organises metadata, and how to inspect it with metadata tables and PyIceberg.

```
catalog pointer ──> metadata.json ──> snapshot ──> manifest list ──> manifest(s) ──> data files (Parquet)
```

| Level | What it holds |
|---|---|
| `metadata.json` | schema, partition spec, properties, the list of snapshots, which one is current |
| snapshot | the state of the table after one commit |
| manifest list / manifest | which data files belong to the snapshot, with per-file column statistics |
| data file | the actual rows, as Parquet |

Commits never modify existing files. They write new files and then atomically swap the catalog pointer to a new `metadata.json`.

In [ ]:
from helpers import get_spark, reset_orders

spark = get_spark("03-under-the-hood")
table = reset_orders(spark, "orders", n=6, commits=3)   # 3 appends = 3 snapshots
spark.table(table).orderBy("order_id").show()

## 1. Metadata tables
Append a metadata table name to any Iceberg table to query its metadata with plain SQL.

In [ ]:
spark.sql("SELECT committed_at, snapshot_id, parent_id, operation FROM demo.course.orders.snapshots").show(truncate=False)

In [ ]:
spark.sql("""
    SELECT snapshot_id, summary['added-records'] AS added_records, summary['total-records'] AS total_records
    FROM demo.course.orders.snapshots ORDER BY committed_at
""").show()

In [ ]:
spark.sql("SELECT file_path, record_count, file_size_in_bytes FROM demo.course.orders.files").show(truncate=60)

In [ ]:
spark.sql("SELECT path, added_snapshot_id, added_data_files_count, existing_data_files_count FROM demo.course.orders.manifests").show(truncate=60)

`history` shows the timeline of which snapshot was current when. We will use it for time travel later.

In [ ]:
spark.sql("SELECT * FROM demo.course.orders.history").show(truncate=False)

## 2. Look at the real files in object storage
The table's files are in the `warehouse` bucket. We list them with PyArrow's S3 file system (RustFS speaks the S3 API).

In [ ]:
from pyarrow import fs

s3 = fs.S3FileSystem(endpoint_override="rustfs:9000", scheme="http",
                     access_key="admin", secret_key="password", region="us-east-1")

def list_table_files(prefix):
    infos = s3.get_file_info(fs.FileSelector(prefix, recursive=True))
    return sorted((i.path.replace(prefix + "/", ""), i.size) for i in infos if i.is_file)

from pyiceberg.catalog import load_catalog
catalog = load_catalog("default")
tbl = catalog.load_table("course.orders")
print("Table location:", tbl.location())
print("Current metadata file:", tbl.metadata_location)

In [ ]:
prefix = tbl.location().replace("s3://", "")
for path, size in list_table_files(prefix):
    print(f"{size:>8}  {path}")

You should see a `data/` folder with Parquet files, and a `metadata/` folder with one `*.metadata.json` per commit,
plus the Avro manifest files (`snap-*.avro`). **Three commits, three metadata files.**

## 3. Read a metadata.json

In [ ]:
import json

with tbl.io.new_input(tbl.metadata_location).open() as f:
    meta = json.load(f)

print("keys:", sorted(meta))
print("format-version:", meta["format-version"])
print("snapshots:", len(meta["snapshots"]))
print("current-snapshot-id:", meta["current-snapshot-id"])

### Your turn

Use the `files` metadata table to find the **number of data files** and the **total size in bytes** of the table.
Assert that the file count equals the number of snapshots (each append wrote one small file).

In [ ]:
# Write your solution here


<details>
<summary>Show solution</summary>

```python
stats = spark.sql(
    "SELECT COUNT(*) AS files, SUM(file_size_in_bytes) AS bytes FROM demo.course.orders.files"
).collect()[0]
snaps = spark.sql("SELECT COUNT(*) FROM demo.course.orders.snapshots").collect()[0][0]
print(stats)
assert stats["files"] == snaps == 3
```

</details>

### Recap
* A table = `metadata.json` → snapshots → manifests → Parquet data files.
* Commits add files and swap a pointer; old files are never rewritten in place.
* `table.snapshots`, `table.files`, `table.manifests` and `table.history` expose all of it in SQL.
* Many tiny files accumulate with small appends. We fix that in **09 · Maintenance**.

Next: **04 · Updates, deletes and MERGE**.